# albu-tta full ImageNet run on Google Colab

This notebook launches the resumable full ImageNet validation pipeline. Do not put secrets in this notebook. It uses Google Drive for persistent artifacts and `full-run-status --next-command` for orchestration.

## 1. Mount Drive

Approve the browser prompt. The run stores `artifacts/` and `reports/` under Google Drive so Colab disconnects do not lose completed shards.

In [ ]:
from google.colab import drive  # type: ignore[unresolved-import]

drive.mount('/content/drive')

## 2. Configure Paths

Set `IMAGENET_VAL_DIR` to an ImageNet validation folder laid out as `val/class_name/image.JPEG`. The notebook prefers the local prepared folder from the ImageNet download helper and falls back to the Google Drive path.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/dKosarevsky/albu-tta.git'
BRANCH = 'main'
WORKDIR = Path('/content/albu-tta')
CONFIG = WORKDIR / 'configs/experiment/resnet50_a1_in1k.yaml'

LOCAL_IMAGENET_VAL_DIR = Path('/content/imagenet_val_prepare/val')
DRIVE_IMAGENET_VAL_DIR = Path('/content/drive/MyDrive/datasets/imagenet/val')
IMAGENET_VAL_DIR = (
    LOCAL_IMAGENET_VAL_DIR
    if LOCAL_IMAGENET_VAL_DIR.exists()
    else DRIVE_IMAGENET_VAL_DIR
)
DRIVE_RUN_ROOT = Path('/content/drive/MyDrive/albu-tta-runs/resnet50_a1_in1k')

assert str(IMAGENET_VAL_DIR), 'Set IMAGENET_VAL_DIR before launching the run'
print('ImageNet val:', IMAGENET_VAL_DIR)
print('Persistent run root:', DRIVE_RUN_ROOT)

## 3. Clone Repo And Install Dependencies

In [ ]:
import shutil
import subprocess


def run(cmd, cwd=None):
    print('+', ' '.join(str(part) for part in cmd))
    subprocess.run([str(part) for part in cmd], cwd=cwd, check=True)

if not WORKDIR.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, WORKDIR])
else:
    run(['git', 'fetch', 'origin', BRANCH], cwd=WORKDIR)
    run(['git', 'checkout', BRANCH], cwd=WORKDIR)
    run(['git', 'pull', '--ff-only'], cwd=WORKDIR)

run(['python', '-m', 'pip', 'install', '-q', 'uv'])
run(['uv', 'sync', '--extra', 'stackers'], cwd=WORKDIR)

## 4. Link Persistent Output Directories

In [ ]:
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)

for name in ('artifacts', 'reports'):
    persistent = DRIVE_RUN_ROOT / name
    link = WORKDIR / name
    persistent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink() or link.exists():
        if link.is_symlink() or link.is_file():
            link.unlink()
        elif link.is_dir():
            shutil.rmtree(link)
    link.symlink_to(persistent, target_is_directory=True)
    print(link, '->', persistent)

## 5. Check GPU And Dataset

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print('GPU:', torch.cuda.get_device_name(0))
assert IMAGENET_VAL_DIR.exists(), f'Missing ImageNet val dir: {IMAGENET_VAL_DIR}'
jpeg_count = sum(1 for _ in IMAGENET_VAL_DIR.glob('*/*.JPEG'))
assert jpeg_count == 50_000, f'Expected 50000 ImageNet validation JPEG files, found {jpeg_count}'

run([
    'uv', 'run', 'python', '-m', 'learned_tta.cli', 'check-full-run',
    '--config', CONFIG,
    '--imagenet-val-dir', IMAGENET_VAL_DIR,
], cwd=WORKDIR)

## 6. Status Helpers

Run one next command at a time. If Colab disconnects, reconnect and run this section again.

In [ ]:
import shlex

PLACEHOLDER_IMAGENET = '/path/to/imagenet/val'
CACHE_TEACHER_NUM_WORKERS = 2

def status_json():
    completed = subprocess.run(
        [
            'uv', 'run', 'python', '-m', 'learned_tta.cli', 'full-run-status',
            '--config', str(CONFIG), '--format', 'json',
        ],
        cwd=WORKDIR,
        check=True,
        text=True,
        capture_output=True,
    )
    return completed.stdout

def next_command():
    completed = subprocess.run(
        [
            'uv', 'run', 'python', '-m', 'learned_tta.cli', 'full-run-status',
            '--config', str(CONFIG), '--next-command',
        ],
        cwd=WORKDIR,
        check=True,
        text=True,
        capture_output=True,
    )
    command = completed.stdout.strip()
    if command:
        command = command.replace(PLACEHOLDER_IMAGENET, shlex.quote(str(IMAGENET_VAL_DIR)))
    if 'cache-teacher' in command and '--num-workers' not in command:
        command += f' --num-workers {CACHE_TEACHER_NUM_WORKERS}'
    return command

print(status_json())
print('NEXT:', next_command() or 'none')

## 7. Run The Next Missing Step

This may run for a long time during `cache-teacher`. After it finishes, re-run the previous status cell.

In [ ]:
command = next_command()
if not command:
    print(
        'No required commands left. '
        'Check reports/resnet50_a1_in1k/results.md under DRIVE_RUN_ROOT.'
    )
else:
    print('+', command)
    subprocess.run(command, cwd=WORKDIR, shell=True, check=True)